# 03 · GUE Comparison: Zeta vs Poisson vs Wigner Surmise

This notebook compares locally normalized zeta-zero spacings against two standard reference classes:

1. **Poisson / exponential spacings**  
2. **GUE-style spacings via the Wigner surmise**

It also compares a local three-zero neighborhood statistic across these sequences.

## Why this matters

A central theme in numerical work around zeta zeros is that their unfolded spacing statistics look much closer to **GUE-like behavior** than to **Poisson** behavior.

This notebook is still exploratory:

- zeta zeros come from `mpmath`
- unfolding is done with a transparent local moving-average normalization
- GUE is approximated here by the Wigner surmise, not by full random-matrix diagonalization
- no theorem-level claims are made from finite numerics

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Collect nontrivial zeros and build local-normalized spacings

In [ ]:
N = 300
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
gaps = np.diff(t)

window = 15
kernel = np.ones(window) / window
local_means = np.convolve(gaps, kernel, mode="same")

half = window // 2
for i in range(half):
    local_means[i] = gaps[:i + half + 1].mean()
    local_means[-(i + 1)] = gaps[-(i + half + 1):].mean()

zeta_local = gaps / local_means

len(zeta_local), zeta_local.mean(), zeta_local.std()

## Reference controls

### Poisson / exponential

For a Poisson point process, nearest-neighbor spacings are exponentially distributed with mean 1.

### Wigner surmise (GUE-style proxy)

For comparison, we use the standard Wigner-surmise-type density for GUE-like level repulsion:

\[
p(s) = \frac{32}{\pi^2}s^2 e^{-4s^2/\pi},
\qquad s \ge 0.
\]

This is not a full GUE simulation, but it is a standard reference shape for spacing comparisons.

> **Note:** The GUE comparison here uses the Wigner surmise as a proxy, not full random-matrix simulation.

In [ ]:
M = len(zeta_local)

poisson_spacings = rng.exponential(scale=1.0, size=M)

# GUE-like sample via rejection sampling from the Wigner surmise density
def wigner_gue_pdf(s):
    return (32 / np.pi**2) * s**2 * np.exp(-4 * s**2 / np.pi)

def sample_wigner_gue(size, rng, s_max=3.5):
    grid = np.linspace(0, s_max, 2000)
    pdf_max = wigner_gue_pdf(grid).max()
    out = []
    while len(out) < size:
        s = rng.uniform(0, s_max, size=size)
        u = rng.uniform(0, pdf_max, size=size)
        accepted = s[u <= wigner_gue_pdf(s)]
        out.extend(accepted.tolist())
    return np.array(out[:size])

gue_spacings = sample_wigner_gue(M, rng)

poisson_spacings.mean(), gue_spacings.mean()

## Histogram: zeta vs Poisson vs GUE-style reference

In [ ]:
plt.figure(figsize=(8.8, 5.2))
bins = np.linspace(0, 3.5, 28)
plt.hist(zeta_local, bins=bins, density=True, alpha=0.55, label="zeta local norm")
plt.hist(poisson_spacings, bins=bins, density=True, alpha=0.45, label="Poisson / exponential")
plt.hist(gue_spacings, bins=bins, density=True, alpha=0.45, label="GUE via Wigner surmise")
plt.xlabel("spacing")
plt.ylabel("density")
plt.title("Spacing comparison: zeta vs Poisson vs GUE-style reference")
plt.legend()
plt.show()

## Overlay reference curves

We overlay:

- exponential density:
  \[
  p_{\mathrm{Pois}}(s)=e^{-s}
  \]
- GUE Wigner-surmise density:
  \[
  p_{\mathrm{GUE}}(s)=\frac{32}{\pi^2}s^2 e^{-4s^2/\pi}
  \]

In [ ]:
s = np.linspace(0, 3.5, 400)
poisson_pdf = np.exp(-s)
gue_pdf = (32 / np.pi**2) * s**2 * np.exp(-4 * s**2 / np.pi)

plt.figure(figsize=(8.8, 5.2))
plt.hist(zeta_local, bins=np.linspace(0, 3.5, 28), density=True, alpha=0.45, label="zeta local norm")
plt.plot(s, poisson_pdf, linewidth=2, label="Poisson pdf")
plt.plot(s, gue_pdf, linewidth=2, label="GUE Wigner-surmise pdf")
plt.xlabel("spacing")
plt.ylabel("density")
plt.title("Zeta local spacings with Poisson and GUE reference curves")
plt.legend()
plt.show()

## Small-spacing behavior

A useful visual distinction:

- **Poisson** allows many very small spacings
- **GUE-like** statistics show level repulsion near \(s=0\)
- zeta zeros are expected to look more GUE-like than Poisson-like

In [ ]:
small_bins = np.linspace(0, 1.5, 18)

plt.figure(figsize=(8.6, 5.0))
plt.hist(zeta_local, bins=small_bins, density=True, alpha=0.55, label="zeta local norm")
plt.hist(poisson_spacings, bins=small_bins, density=True, alpha=0.45, label="Poisson / exponential")
plt.hist(gue_spacings, bins=small_bins, density=True, alpha=0.45, label="GUE via Wigner surmise")
plt.xlabel("spacing")
plt.ylabel("density")
plt.title("Small-spacing comparison")
plt.legend()
plt.show()

## Local three-zero neighborhood statistic

Using a spacing sequence \(g_n\), define for each center position:

\[
b_n=\frac{\min(g_L,g_R)}{\max(g_L,g_R)},
\]

where \(g_L\) and \(g_R\) are consecutive neighboring gaps.

This is a compact measure of local balance:
- near 1 = more balanced neighborhood
- near 0 = more uneven neighborhood

In [ ]:
def local_balance(seq):
    left = seq[:-1]
    right = seq[1:]
    return np.minimum(left, right) / np.maximum(left, right)

zeta_balance = local_balance(zeta_local)
poisson_balance = local_balance(poisson_spacings)
gue_balance = local_balance(gue_spacings)

zeta_balance.mean(), poisson_balance.mean(), gue_balance.mean()

## Balance histogram comparison

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 1, 24)
plt.hist(zeta_balance, bins=bins, density=True, alpha=0.55, label="zeta balance")
plt.hist(poisson_balance, bins=bins, density=True, alpha=0.45, label="Poisson balance")
plt.hist(gue_balance, bins=bins, density=True, alpha=0.45, label="GUE balance")
plt.xlabel(r"$b_n = \min(g_L,g_R)/\max(g_L,g_R)$")
plt.ylabel("density")
plt.title("Local balance comparison")
plt.legend()
plt.show()

## Running balance series for zeta

In [ ]:
center_index = np.arange(2, len(t))

plt.figure(figsize=(8.4, 4.8))
plt.plot(center_index, zeta_balance, marker="o", linewidth=1)
plt.axhline(zeta_balance.mean(), linestyle="--", linewidth=1, label=f"mean ≈ {zeta_balance.mean():.3f}")
plt.xlabel("center zero index n")
plt.ylabel("local balance")
plt.title("Zeta local three-zero balance")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

## Simple quantitative summaries

In [ ]:
def summary_stats(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

summary = {
    "zeta_spacing": summary_stats(zeta_local),
    "poisson_spacing": summary_stats(poisson_spacings),
    "gue_spacing": summary_stats(gue_spacings),
    "zeta_balance": summary_stats(zeta_balance),
    "poisson_balance": summary_stats(poisson_balance),
    "gue_balance": summary_stats(gue_balance),
}
summary

## Notes

- The zeta spacing histogram should look visibly less Poisson-like near very small spacings.
- The Wigner-surmise curve gives a useful first GUE-style reference.
- The local balance statistic is not a standard zeta diagnostic, but it offers a compact way to compare local neighborhood asymmetry across different spacing ensembles.
- The GUE reference here is based on the Wigner surmise, not full random-matrix simulation.

## Next directions

A natural next notebook could do one of these:

1. compare larger zero samples  
2. compute pair-correlation-style quantities  
3. generate full random Hermitian matrices and compare empirical eigenvalue spacings  
4. define richer local descriptors using both gaps and nearby values of \(|\zeta(1/2+it)|\)